# Notebook 1: Data Ingestion & Cleaning

**Goal:** Load the Spotify dataset, understand its shape, fix issues, and save a clean version.

**Dataset:** 114,000 Spotify tracks with 21 audio features from Kaggle.

In [1]:
import pandas as pd
import numpy as np
import os

print('Libraries loaded ✓')

Libraries loaded ✓


## 1. Download the dataset

In [2]:
import subprocess

os.makedirs('../data', exist_ok=True)

if not os.path.exists('../data/dataset.csv'):
    print('Downloading dataset...')
    subprocess.run([
        'kaggle', 'datasets', 'download',
        'maharshipandya/-spotify-tracks-dataset',
        '--unzip', '-p', '../data'
    ])
    print('Download complete ✓')
else:
    print('Dataset already exists ✓')

Dataset URL: https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset
License(s): ODbL-1.0


 61%|██████    | 5.00M/8.17M [00:01<00:00, 4.26MB/s]


Download complete ✓


100%|██████████| 8.17M/8.17M [00:01<00:00, 4.79MB/s]


## 2. Load and inspect

In [3]:
df = pd.read_csv('../data/dataset.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

Shape: (114000, 21)
Columns: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,...,-6.746,0,0.1430,0.0322,0.000001,0.358,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.166,...,-17.235,1,0.0763,0.9240,0.000006,0.101,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,...,-9.734,1,0.0557,0.2100,0.000000,0.117,0.120,76.332,4,acoustic


In [4]:
# Basic info — dtypes and null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          11

In [5]:
# Summary statistics — get a feel for distributions
df.describe().round(3)

,Unnamed: 0,popularity,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
count,114000.00,114000.000,114000.000,114000.000,114000.000,114000.000,114000.000,114000.000,114000.000,114000.000,114000.000,114000.000,114000.000,114000.000,114000.000
mean,56999.50,33.239,228029.153,0.567,0.641,5.309,-8.259,0.638,0.085,0.315,0.156,0.214,0.474,122.148,3.904
std,32909.11,22.305,107297.713,0.174,0.252,3.560,5.029,0.481,0.106,0.333,0.310,0.190,0.259,29.978,0.433
min,0.00,0.000,0.000,0.000,0.000,0.000,-49.531,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,28499.75,17.000,174066.000,0.456,0.472,2.000,-10.013,0.000,0.036,0.017,0.000,0.098,0.260,99.219,4.000
50%,56999.50,35.000,212906.000,0.580,0.685,5.000,-7.004,1.000,0.049,0.169,0.000,0.132,0.464,122.017,4.000
75%,85499.25,50.000,261506.000,0.695,0.854,8.000,-5.003,1.000,0.084,0.598,0.049,0.273,0.683,140.071,4.000
max,113999.00,100.000,5237295.000,0.985,1.000,11.000,4.532,1.000,0.965,0.996,1.000,1.000,0.995,243.372,5.000


## 3. Check for issues

In [6]:
# Missing values
missing = df.isnull().sum()
missing[missing > 0]

artists       1
album_name    1
track_name    1
dtype: int64

In [7]:
# Duplicates
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Duplicate track IDs: {df["track_id"].duplicated().sum()}')

Duplicate rows: 0
Duplicate track IDs: 24259


In [8]:
# Target variable distribution
print('Popularity distribution:')
print(df['popularity'].describe())
print(f'\nZero-popularity tracks: {(df["popularity"] == 0).sum()} ({(df["popularity"] == 0).mean():.1%})')

Popularity distribution:
count    114000.000000
mean         33.238535
std          22.305078
min           0.000000
25%          17.000000
50%          35.000000
75%          50.000000
max         100.000000
Name: popularity, dtype: float64

Zero-popularity tracks: 16020 (14.1%)


## 4. Clean

In [9]:
df_clean = df.copy()

# Drop exact duplicates
df_clean = df_clean.drop_duplicates(subset='track_id')

# Drop tracks with zero popularity — likely unlisted/unreleased
df_clean = df_clean[df_clean['popularity'] > 0]

# Drop rows with nulls in audio features
audio_features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'duration_ms'
]
df_clean = df_clean.dropna(subset=audio_features)

# Convert duration from ms to seconds (more interpretable)
df_clean['duration_s'] = (df_clean['duration_ms'] / 1000).round(1)

# Create binary target: 1 = hit (top 25% popularity), 0 = not
threshold = df_clean['popularity'].quantile(0.75)
df_clean['is_hit'] = (df_clean['popularity'] >= threshold).astype(int)

print(f'Clean dataset shape: {df_clean.shape}')
print(f'Hit threshold (top 25%): popularity >= {threshold:.0f}')
print(f'Class balance — hits: {df_clean["is_hit"].mean():.1%}')

Clean dataset shape: (80293, 23)
Hit threshold (top 25%): popularity >= 50
Class balance — hits: 26.4%


## 5. Save clean data

In [10]:
df_clean.to_csv('../data/spotify_clean.csv', index=False)
print(f'Saved → data/spotify_clean.csv ({df_clean.shape[0]:,} rows)')

Saved → data/spotify_clean.csv (80,293 rows)


---
## Summary

| Step | Result |
|---|---|
| Raw rows | 114,000 |
| After dedup + filter | ~89,000 |
| Target | `is_hit` (binary, 25% positive) |
| Features | 10 audio features + genre + explicit |

**Next:** `02_eda.ipynb` — explore what separates hits from non-hits.